In [2]:
!pip install dash dash-bootstrap-components pandas plotly

   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/7.9 MB 2.6 MB/s eta 0:00:04
   -- ------------------------------------- 0.6/7.9 MB 6.1 MB/s eta 0:00:02
   ------- -------------------------------- 1.5/7.9 MB 11.7 MB/s eta 0:00:01
   ------------ --------------------------- 2.4/7.9 MB 12.8 MB/s eta 0:00:01
   ----------------- ---------------------- 3.5/7.9 MB 14.9 MB/s eta 0:00:01
   --------------------- ------------------ 4.4/7.9 MB 15.4 MB/s eta 0:00:01
   --------------------------- ------------ 5.5/7.9 MB 16.7 MB/s eta 0:00:01
   --------------------------------- ------ 6.7/7.9 MB 17.7 MB/s eta 0:00:01
   ---------------------------------------  7.8/7.9 MB 18.3 MB/s eta 0:00:01
   ---------------------------------------- 7.9/7.9 MB 17.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/202.5 kB ? eta -:--:--
   --------------------------------------- 202.5/202.5 kB 12.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc
import dash_bootstrap_components as dbc
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# Read CSV with warning prevention
df = pd.read_csv('Airbnb_Open_Data.csv', low_memory=False)

# Clean and filter for map chart
df['price_clean'] = pd.to_numeric(df['price'].replace('[\$,]', '', regex=True), errors='coerce')
df['service_fee_clean'] = df['service fee'].replace('[\$,]', '', regex=True).astype(float)
df['last review'] = pd.to_datetime(df['last review'], errors='coerce')
df_map = df.dropna(subset=['lat', 'long', 'price_clean']) 

# Aggregate data
avg_price = df.groupby('room type')['price_clean'].mean().reset_index()
monthly_reviews = df.set_index('last review').resample('M')['number of reviews'].sum().reset_index()

# Initialize Dash app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Define layout
app.layout = dbc.Container([
    html.H1("Airbnb Data Dashboard", className="text-center my-4"),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=px.histogram(df, x='neighbourhood group', title='Listings by Region', color_discrete_sequence=['indianred'])), md=6),
        dbc.Col(dcc.Graph(figure=px.pie(df, names='room type', title='Room Type Distribution', color_discrete_sequence=px.colors.sequential.RdBu)), md=6)
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=px.scatter(df, x='review rate number', y='price_clean', title='Price vs Review Rating', color='room type', opacity=0.6, color_discrete_sequence=px.colors.qualitative.Set1)), md=6),
        dbc.Col(dcc.Graph(figure=px.bar(avg_price, x='room type', y='price_clean', title='Average Price by Room Type', color='price_clean', color_continuous_scale='Agsunset')), md=6)
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=px.scatter_mapbox(
    df_map,
    lat="lat", lon="long",
    color="room type",
    size="price_clean",
    size_max=10, zoom=10,
    mapbox_style="carto-positron",
    title="Geographic Distribution of Listings"
)), md=12)

    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=px.line(monthly_reviews, x='last review', y='number of reviews', title='Monthly Trend of Reviews Over Time')), md=6),
        dbc.Col(dcc.Graph(figure=px.histogram(df, x='calculated host listings count', title='Distribution of Listings per Host', nbins=50, color_discrete_sequence=['mediumseagreen'])), md=6)
    ])

], fluid=True)

# Run the app
if __name__ == '__main__':
    app.run(debug=True)
